<a href="https://colab.research.google.com/github/DenisSakaj/machine-learning-colab/blob/main/CN6005_Grad_CAM%2C_LIME%2C_Confusion_Matrixes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -----------------------------
# 1️⃣ Import Libraries
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Input
import tensorflow as tf
from lime import lime_tabular, lime_image
from skimage.segmentation import mark_boundaries

# -----------------------------
# 2️⃣ Load Fashion_MNIST
# -----------------------------
from tensorflow.keras.datasets import fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# Flatten for Logistic Regression / ANN
X_train_flat = x_train.reshape(-1, 28*28)
X_test_flat  = x_test.reshape(-1, 28*28)

# Reshape for CNN
X_train_cnn = x_train.reshape(-1,28,28,1)
X_test_cnn  = x_test.reshape(-1,28,28,1)

# -----------------------------
# 3️⃣ Logistic Regression
# -----------------------------
log_reg = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial')
log_reg.fit(X_train_flat, y_train)
y_pred_lr = log_reg.predict(X_test_flat)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {acc_lr:.4f}")

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(6,5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -----------------------------
# 4️⃣ ANN (MLP)
# -----------------------------
ann_model = Sequential([
    Input(shape=(784,)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])
ann_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
ann_model.fit(X_train_flat, y_train, epochs=10, batch_size=32, validation_split=0.1, verbose=0)

y_pred_ann = np.argmax(ann_model.predict(X_test_flat), axis=1)
acc_ann = accuracy_score(y_test, y_pred_ann)
print(f"ANN Accuracy: {acc_ann:.4f}")

# Confusion matrix
cm_ann = confusion_matrix(y_test, y_pred_ann)
plt.figure(figsize=(6,5))
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - ANN")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -----------------------------
# 5️⃣ CNN
# -----------------------------
cnn_model = Sequential([
    Input(shape=(28,28,1)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_split=0.1, verbose=0)

y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn), axis=1)
acc_cnn = accuracy_score(y_test, y_pred_cnn)
print(f"CNN Accuracy: {acc_cnn:.4f}")

# Confusion matrix
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(6,5))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - CNN")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -----------------------------
# 6️⃣ LIME Explainability
# -----------------------------
idx = 10  # test sample index

# --- Logistic Regression & ANN (Tabular LIME) ---
explainer_tab = lime_tabular.LimeTabularExplainer(
    training_data=X_train_flat,
    mode='classification',
    feature_names=[f'pixel_{i}' for i in range(784)],
    class_names=[str(i) for i in range(10)],
    verbose=False
)

# Logistic Regression
exp_lr = explainer_tab.explain_instance(X_test_flat[idx], log_reg.predict_proba)
exp_lr.show_in_notebook()

# ANN
exp_ann = explainer_tab.explain_instance(X_test_flat[idx], ann_model.predict)
exp_ann.show_in_notebook()

# --- CNN (Image LIME) ---
sample_img = X_test_cnn[idx]
sample_rgb = np.repeat(sample_img, 3, axis=2)  # fake RGB

def cnn_predict(images):
    return cnn_model.predict(images[:,:,:,0:1])  # convert back to 1-channel

explainer_img = lime_image.LimeImageExplainer()
exp_cnn = explainer_img.explain_instance(
    image=sample_rgb,
    classifier_fn=cnn_predict,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

top_label = exp_cnn.top_labels[0]
temp, mask = exp_cnn.get_image_and_mask(top_label, positive_only=True, num_features=10, hide_rest=False)
plt.figure(figsize=(4,4))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation - CNN")
plt.axis("off")